In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
pip install torchviz

In [ ]:
import os
import glob
import json
import copy
import random
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt

from sklearn.metrics import accuracy_score, f1_score, classification_report

# =========================================================
# CONFIG
# =========================================================
DATA_DIR = "/content/drive/MyDrive/processed_meg_regression"

BATCH_SIZE = 8
EPOCHS = 100
LR = 1e-4
WEIGHT_DECAY = 1e-4
RUN_SEED = 42

WINDOW_SIZE = 2048
WINDOW_STRIDE = 1024

EARLY_STOPPING_PATIENCE = 12
MIN_DELTA = 1e-4

# -------- multitask weighting --------
VALENCE_LOSS_WEIGHT = 1.5
AROUSAL_LOSS_WEIGHT = 1.0

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

torch.manual_seed(RUN_SEED)
np.random.seed(RUN_SEED)
random.seed(RUN_SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

# =========================================================
# JSON HELPER
# =========================================================
def to_python(obj):
    if isinstance(obj, dict):
        return {k: to_python(v) for k, v in obj.items()}
    elif isinstance(obj, list):
        return [to_python(v) for v in obj]
    elif isinstance(obj, tuple):
        return [to_python(v) for v in obj]
    elif isinstance(obj, np.integer):
        return int(obj)
    elif isinstance(obj, np.floating):
        return float(obj)
    elif isinstance(obj, np.ndarray):
        return obj.tolist()
    else:
        return obj

# =========================================================
# WINDOWING
# =========================================================
def extract_windows_from_trial(trial_x, trial_y, window_size=2048, stride=1024):
    c, t = trial_x.shape
    xs = []
    start = 0

    while start + window_size <= t:
        xs.append(trial_x[:, start:start + window_size])
        start += stride

    if len(xs) == 0:
        return None, None

    return np.stack(xs).astype(np.float32), np.array(trial_y, dtype=np.float32)

# =========================================================
# LOADING
# =========================================================
def extract_subject_id_from_path(x_path):
    base = os.path.basename(x_path)
    return base.split("_")[0]

def load_all_trials_with_subjects(data_dir):
    x_paths = sorted(glob.glob(os.path.join(data_dir, "*_X.npy")))
    print(f"Searching in: {data_dir}")
    print(f"Found {len(x_paths)} X files")

    seq_x_list, y_list, subject_list = [], [], []

    for x_path in x_paths:
        y_path = x_path.replace("_X.npy", "_y.npy")
        if not os.path.exists(y_path):
            print(f"[WARN] Missing y file for: {x_path}")
            continue

        subject_id = extract_subject_id_from_path(x_path)
        X = np.load(x_path)
        y = np.load(y_path)

        n = min(len(X), len(y))
        X, y = X[:n], y[:n]

        for i in range(n):
            seq_x, y_trial = extract_windows_from_trial(
                X[i], y[i],
                window_size=WINDOW_SIZE,
                stride=WINDOW_STRIDE
            )
            if seq_x is not None:
                seq_x_list.append(seq_x)
                y_list.append(y_trial)
                subject_list.append(subject_id)

    print(f"Total usable trial sequences: {len(seq_x_list)}")
    print(f"Total unique subjects: {len(set(subject_list))}")
    return seq_x_list, y_list, subject_list

def filter_by_subjects(seq_x_list, y_list, subject_list, allowed_subjects):
    out_x, out_y = [], []
    for x, y, s in zip(seq_x_list, y_list, subject_list):
        if s in allowed_subjects:
            out_x.append(x)
            out_y.append(y)
    return out_x, out_y

# =========================================================
# LABEL CONVERSION
# =========================================================
def compute_train_thresholds(train_y_list):
    Y_train = np.stack(train_y_list, axis=0)
    valence_thresh = np.median(Y_train[:, 0])
    arousal_thresh = np.median(Y_train[:, 1])
    return float(valence_thresh), float(arousal_thresh)

def make_binary_labels_from_threshold(y_list, valence_thresh, arousal_thresh):
    """
    y_list: list of [2] arrays => [valence, arousal]
    """
    Y = np.stack(y_list, axis=0)
    valence_labels = (Y[:, 0] >= valence_thresh).astype(np.int64)
    arousal_labels = (Y[:, 1] >= arousal_thresh).astype(np.int64)
    return valence_labels, arousal_labels

# =========================================================
# DATASET
# =========================================================
class TrialMultiTaskDataset(Dataset):
    def __init__(self, seq_x_list, valence_labels, arousal_labels):
        self.seq_x_list = seq_x_list
        self.valence_labels = valence_labels.astype(np.int64)
        self.arousal_labels = arousal_labels.astype(np.int64)

    def __len__(self):
        return len(self.seq_x_list)

    def __getitem__(self, idx):
        x_seq = torch.tensor(self.seq_x_list[idx], dtype=torch.float32)
        y_val = torch.tensor(self.valence_labels[idx], dtype=torch.long)
        y_aro = torch.tensor(self.arousal_labels[idx], dtype=torch.long)
        return x_seq, y_val, y_aro

# =========================================================
# NORMALIZATION
# =========================================================
def compute_train_stats(dataset_subset):
    xs = []

    for x_seq, _, _ in dataset_subset:
        xs.append(x_seq.numpy())

    X = np.concatenate(xs, axis=0)  # (all_windows, C, T)

    x_mean = X.mean(axis=(0, 2), keepdims=True)
    x_std = X.std(axis=(0, 2), keepdims=True) + 1e-6

    return x_mean.astype(np.float32), x_std.astype(np.float32)

class NormalizedMultiTaskDataset(Dataset):
    def __init__(self, base_dataset, x_mean, x_std):
        self.base_dataset = base_dataset
        self.x_mean = torch.tensor(x_mean, dtype=torch.float32)
        self.x_std = torch.tensor(x_std, dtype=torch.float32)

    def __len__(self):
        return len(self.base_dataset)

    def __getitem__(self, idx):
        x_seq, y_val, y_aro = self.base_dataset[idx]

        x_seq = (x_seq - self.x_mean) / self.x_std
        x_seq = x_seq - x_seq.mean(dim=-1, keepdim=True)
        x_seq = x_seq / (x_seq.std(dim=-1, keepdim=True) + 1e-6)
        x_seq = x_seq / (x_seq.abs().max(dim=-1, keepdim=True)[0] + 1e-6)

        return x_seq, y_val, y_aro

# =========================================================
# MODEL
# =========================================================
class PlainCNNEncoder(nn.Module):
    def __init__(self, in_channels=143, feature_dim=256, dropout=0.2):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv1d(in_channels, 64, kernel_size=7, padding=3),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.MaxPool1d(2),

            nn.Conv1d(64, 128, kernel_size=5, padding=2),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.MaxPool1d(2),

            nn.Conv1d(128, feature_dim, kernel_size=3, padding=1),
            nn.BatchNorm1d(feature_dim),
            nn.ReLU(),
            nn.AdaptiveAvgPool1d(1)
        )
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        x = self.net(x)
        x = x.squeeze(-1)
        x = self.dropout(x)
        return x

class MultiTaskBottleneckCNNBiLSTM(nn.Module):
    def __init__(self, in_channels=143, cnn_feature_dim=256, lstm_hidden=256, dropout=0.2):
        super().__init__()

        self.encoder = PlainCNNEncoder(
            in_channels=in_channels,
            feature_dim=cnn_feature_dim,
            dropout=dropout
        )

        self.temporal_bilstm = nn.LSTM(
            input_size=cnn_feature_dim,
            hidden_size=lstm_hidden,
            num_layers=1,
            batch_first=True,
            bidirectional=True
        )

        shared_dim = lstm_hidden * 2

        self.shared_proj = nn.Sequential(
            nn.Linear(shared_dim, 256),
            nn.ReLU(),
            nn.Dropout(dropout)
        )

        # task-specific bottlenecks
        self.valence_bottleneck = nn.Sequential(
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(dropout)
        )

        self.arousal_bottleneck = nn.Sequential(
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(dropout)
        )

        self.valence_head = nn.Linear(128, 2)
        self.arousal_head = nn.Linear(128, 2)

    def forward(self, x_seq):
        # x_seq: [B, W, C, T]
        B, W, C, T = x_seq.shape

        x = x_seq.reshape(B * W, C, T)
        z = self.encoder(x)            # [B*W, F]
        z = z.reshape(B, W, -1)        # [B, W, F]

        z_lstm, _ = self.temporal_bilstm(z)   # [B, W, 2H]
        z_pool = z_lstm.mean(dim=1)           # [B, 2H]

        shared = self.shared_proj(z_pool)     # [B, 256]

        val_feat = self.valence_bottleneck(shared)
        aro_feat = self.arousal_bottleneck(shared)

        val_logits = self.valence_head(val_feat)
        aro_logits = self.arousal_head(aro_feat)

        return val_logits, aro_logits

# =========================================================
# METRICS
# =========================================================
def compute_binary_metrics(y_true, y_pred):
    acc = accuracy_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred, average="binary", zero_division=0)
    return acc, f1

# =========================================================
# TRAIN / EVAL
# =========================================================
def train_one_epoch(model, loader, optimizer, criterion_val, criterion_aro):
    model.train()

    total_loss = 0.0
    total_val_loss = 0.0
    total_aro_loss = 0.0

    for x_seq, y_val, y_aro in loader:
        x_seq = x_seq.to(DEVICE)
        y_val = y_val.to(DEVICE)
        y_aro = y_aro.to(DEVICE)

        val_logits, aro_logits = model(x_seq)

        loss_val = criterion_val(val_logits, y_val)
        loss_aro = criterion_aro(aro_logits, y_aro)

        loss = VALENCE_LOSS_WEIGHT * loss_val + AROUSAL_LOSS_WEIGHT * loss_aro

        optimizer.zero_grad()
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        total_loss += loss.item()
        total_val_loss += loss_val.item()
        total_aro_loss += loss_aro.item()

    n_batches = len(loader)
    return {
        "loss": total_loss / n_batches,
        "valence_loss": total_val_loss / n_batches,
        "arousal_loss": total_aro_loss / n_batches,
    }

@torch.no_grad()
def evaluate(model, loader, criterion_val, criterion_aro):
    model.eval()

    total_loss = 0.0
    total_val_loss = 0.0
    total_aro_loss = 0.0

    all_val_true, all_val_pred = [], []
    all_aro_true, all_aro_pred = [], []

    for x_seq, y_val, y_aro in loader:
        x_seq = x_seq.to(DEVICE)
        y_val = y_val.to(DEVICE)
        y_aro = y_aro.to(DEVICE)

        val_logits, aro_logits = model(x_seq)

        loss_val = criterion_val(val_logits, y_val)
        loss_aro = criterion_aro(aro_logits, y_aro)
        loss = VALENCE_LOSS_WEIGHT * loss_val + AROUSAL_LOSS_WEIGHT * loss_aro

        total_loss += loss.item()
        total_val_loss += loss_val.item()
        total_aro_loss += loss_aro.item()

        val_pred = torch.argmax(val_logits, dim=1)
        aro_pred = torch.argmax(aro_logits, dim=1)

        all_val_true.append(y_val.cpu().numpy())
        all_val_pred.append(val_pred.cpu().numpy())

        all_aro_true.append(y_aro.cpu().numpy())
        all_aro_pred.append(aro_pred.cpu().numpy())

    y_val_true = np.concatenate(all_val_true)
    y_val_pred = np.concatenate(all_val_pred)

    y_aro_true = np.concatenate(all_aro_true)
    y_aro_pred = np.concatenate(all_aro_pred)

    val_acc, val_f1 = compute_binary_metrics(y_val_true, y_val_pred)
    aro_acc, aro_f1 = compute_binary_metrics(y_aro_true, y_aro_pred)

    mean_f1 = 0.5 * (val_f1 + aro_f1)

    # valence-biased model selection score
    weighted_score = 0.7 * val_f1 + 0.3 * aro_f1

    return {
        "loss": total_loss / len(loader),
        "valence_loss": total_val_loss / len(loader),
        "arousal_loss": total_aro_loss / len(loader),

        "valence_accuracy": val_acc,
        "valence_f1": val_f1,
        "arousal_accuracy": aro_acc,
        "arousal_f1": aro_f1,

        "mean_f1": mean_f1,
        "weighted_score": weighted_score,

        "y_val_true": y_val_true,
        "y_val_pred": y_val_pred,
        "y_aro_true": y_aro_true,
        "y_aro_pred": y_aro_pred,
    }

# =========================================================
# CURVE PLOTTING
# =========================================================
def save_fold_curves(history, held_out_subject):
    epochs = range(1, len(history["train_loss"]) + 1)

    plt.figure(figsize=(8, 5))
    plt.plot(epochs, history["train_loss"], label="Train Total Loss")
    plt.plot(epochs, history["val_loss"], label="Val Total Loss")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title(f"Multitask Bottleneck Loss - Held-out {held_out_subject}")
    plt.legend()
    plt.tight_layout()
    plt.savefig(f"curve_loss_loso_multitask_bottleneck_{held_out_subject}.png")
    plt.close()

    plt.figure(figsize=(8, 5))
    plt.plot(epochs, history["val_valence_f1"], label="Valence F1")
    plt.plot(epochs, history["val_arousal_f1"], label="Arousal F1")
    plt.plot(epochs, history["val_mean_f1"], label="Mean F1")
    plt.plot(epochs, history["val_weighted_score"], label="Weighted Score")
    plt.xlabel("Epoch")
    plt.ylabel("Metric")
    plt.title(f"Multitask Bottleneck Metrics - Held-out {held_out_subject}")
    plt.legend()
    plt.tight_layout()
    plt.savefig(f"curve_metrics_loso_multitask_bottleneck_{held_out_subject}.png")
    plt.close()

# =========================================================
# TRAIN ONE LOSO FOLD
# =========================================================
def train_one_loso_fold_multitask_bottleneck(held_out_subject, seq_x_list, y_list, subject_list):
    train_subjects = sorted(list(set(subject_list) - {held_out_subject}))
    val_subjects = [held_out_subject]

    train_x, train_y_raw = filter_by_subjects(seq_x_list, y_list, subject_list, set(train_subjects))
    val_x, val_y_raw = filter_by_subjects(seq_x_list, y_list, subject_list, set(val_subjects))

    valence_thresh, arousal_thresh = compute_train_thresholds(train_y_raw)

    train_val_labels, train_aro_labels = make_binary_labels_from_threshold(
        train_y_raw, valence_thresh, arousal_thresh
    )
    val_val_labels, val_aro_labels = make_binary_labels_from_threshold(
        val_y_raw, valence_thresh, arousal_thresh
    )

    print("\n" + "=" * 80)
    print(f"LOSO FOLD | Held-out subject: {held_out_subject} | MULTITASK BOTTLENECK")
    print("=" * 80)
    print(f"Train trials: {len(train_x)}")
    print(f"Val trials:   {len(val_x)}")
    print(f"Train threshold valence: {valence_thresh:.4f}")
    print(f"Train threshold arousal: {arousal_thresh:.4f}")
    print(f"Train valence counts: {np.bincount(train_val_labels, minlength=2)}")
    print(f"Val   valence counts: {np.bincount(val_val_labels, minlength=2)}")
    print(f"Train arousal counts: {np.bincount(train_aro_labels, minlength=2)}")
    print(f"Val   arousal counts: {np.bincount(val_aro_labels, minlength=2)}")

    train_base = TrialMultiTaskDataset(train_x, train_val_labels, train_aro_labels)
    val_base = TrialMultiTaskDataset(val_x, val_val_labels, val_aro_labels)

    x_mean, x_std = compute_train_stats(train_base)
    train_set = NormalizedMultiTaskDataset(train_base, x_mean, x_std)
    val_set = NormalizedMultiTaskDataset(val_base, x_mean, x_std)

    train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True, num_workers=0, pin_memory=True)
    val_loader = DataLoader(val_set, batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True)

    model = MultiTaskBottleneckCNNBiLSTM(
        in_channels=143,
        cnn_feature_dim=256,
        lstm_hidden=256,
        dropout=0.2
    ).to(DEVICE)

    val_class_counts = np.bincount(train_val_labels, minlength=2).astype(np.float32)
    aro_class_counts = np.bincount(train_aro_labels, minlength=2).astype(np.float32)

    val_class_weights = val_class_counts.sum() / (2.0 * np.maximum(val_class_counts, 1.0))
    aro_class_weights = aro_class_counts.sum() / (2.0 * np.maximum(aro_class_counts, 1.0))

    val_class_weights = torch.tensor(val_class_weights, dtype=torch.float32, device=DEVICE)
    aro_class_weights = torch.tensor(aro_class_weights, dtype=torch.float32, device=DEVICE)

    criterion_val = nn.CrossEntropyLoss(weight=val_class_weights)
    criterion_aro = nn.CrossEntropyLoss(weight=aro_class_weights)

    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode="max",
        factor=0.5,
        patience=5
    )

    best_score = -float("inf")
    best_epoch = -1
    best_checkpoint = None
    epochs_without_improvement = 0

    history = {
        "train_loss": [],
        "train_valence_loss": [],
        "train_arousal_loss": [],

        "val_loss": [],
        "val_valence_loss": [],
        "val_arousal_loss": [],

        "val_valence_acc": [],
        "val_valence_f1": [],
        "val_arousal_acc": [],
        "val_arousal_f1": [],
        "val_mean_f1": [],
        "val_weighted_score": [],
    }

    for epoch in range(1, EPOCHS + 1):
        train_metrics = train_one_epoch(model, train_loader, optimizer, criterion_val, criterion_aro)
        val_metrics = evaluate(model, val_loader, criterion_val, criterion_aro)

        score = val_metrics["weighted_score"]
        scheduler.step(score)

        history["train_loss"].append(train_metrics["loss"])
        history["train_valence_loss"].append(train_metrics["valence_loss"])
        history["train_arousal_loss"].append(train_metrics["arousal_loss"])

        history["val_loss"].append(val_metrics["loss"])
        history["val_valence_loss"].append(val_metrics["valence_loss"])
        history["val_arousal_loss"].append(val_metrics["arousal_loss"])

        history["val_valence_acc"].append(val_metrics["valence_accuracy"])
        history["val_valence_f1"].append(val_metrics["valence_f1"])
        history["val_arousal_acc"].append(val_metrics["arousal_accuracy"])
        history["val_arousal_f1"].append(val_metrics["arousal_f1"])
        history["val_mean_f1"].append(val_metrics["mean_f1"])
        history["val_weighted_score"].append(val_metrics["weighted_score"])

        current_lr = optimizer.param_groups[0]["lr"]

        print(
            f"Held-out {held_out_subject} | Epoch {epoch:03d} | "
            f"LR: {current_lr:.6f} | "
            f"Train Loss: {train_metrics['loss']:.4f} | "
            f"Val Loss: {val_metrics['loss']:.4f} | "
            f"Valence Acc: {val_metrics['valence_accuracy']:.4f} | "
            f"Valence F1: {val_metrics['valence_f1']:.4f} | "
            f"Arousal Acc: {val_metrics['arousal_accuracy']:.4f} | "
            f"Arousal F1: {val_metrics['arousal_f1']:.4f} | "
            f"Mean F1: {val_metrics['mean_f1']:.4f} | "
            f"Weighted Score: {val_metrics['weighted_score']:.4f}"
        )

        improved = score > (best_score + MIN_DELTA)

        if improved:
            best_score = score
            best_epoch = epoch
            epochs_without_improvement = 0

            best_checkpoint = {
                "held_out_subject": held_out_subject,
                "epoch": epoch,
                "weighted_score": val_metrics["weighted_score"],
                "mean_f1": val_metrics["mean_f1"],

                "valence_accuracy": val_metrics["valence_accuracy"],
                "valence_f1": val_metrics["valence_f1"],
                "arousal_accuracy": val_metrics["arousal_accuracy"],
                "arousal_f1": val_metrics["arousal_f1"],

                "y_val_true": val_metrics["y_val_true"],
                "y_val_pred": val_metrics["y_val_pred"],
                "y_aro_true": val_metrics["y_aro_true"],
                "y_aro_pred": val_metrics["y_aro_pred"],

                "valence_thresh": valence_thresh,
                "arousal_thresh": arousal_thresh,
            }

            torch.save(
                {
                    "held_out_subject": held_out_subject,
                    "epoch": epoch,
                    "model_state_dict": copy.deepcopy(model.state_dict()),
                    "weighted_score": val_metrics["weighted_score"],
                    "mean_f1": val_metrics["mean_f1"],
                    "valence_accuracy": val_metrics["valence_accuracy"],
                    "valence_f1": val_metrics["valence_f1"],
                    "arousal_accuracy": val_metrics["arousal_accuracy"],
                    "arousal_f1": val_metrics["arousal_f1"],
                    "valence_thresh": valence_thresh,
                    "arousal_thresh": arousal_thresh,
                },
                f"best_model_loso_multitask_bottleneck_{held_out_subject}.pt"
            )

            print(
                f"🔥 Best multitask bottleneck fold updated | Epoch {epoch} | "
                f"Valence F1: {val_metrics['valence_f1']:.4f} | "
                f"Arousal F1: {val_metrics['arousal_f1']:.4f} | "
                f"Weighted Score: {val_metrics['weighted_score']:.4f}"
            )
        else:
            epochs_without_improvement += 1

        if epochs_without_improvement >= EARLY_STOPPING_PATIENCE:
            print(
                f"Early stopping for held-out {held_out_subject} at epoch {epoch}. "
                f"Best epoch was {best_epoch}."
            )
            break

    if best_checkpoint is None:
        raise RuntimeError(f"No best checkpoint saved for held-out subject {held_out_subject}")

    np.save(f"best_y_true_loso_multitask_bottleneck_valence_{held_out_subject}.npy", best_checkpoint["y_val_true"])
    np.save(f"best_y_pred_loso_multitask_bottleneck_valence_{held_out_subject}.npy", best_checkpoint["y_val_pred"])
    np.save(f"best_y_true_loso_multitask_bottleneck_arousal_{held_out_subject}.npy", best_checkpoint["y_aro_true"])
    np.save(f"best_y_pred_loso_multitask_bottleneck_arousal_{held_out_subject}.npy", best_checkpoint["y_aro_pred"])

    save_fold_curves(history, held_out_subject)

    with open(f"history_loso_multitask_bottleneck_{held_out_subject}.json", "w") as f:
        json.dump(to_python(history), f, indent=4)

    print("\nVALENCE classification report:")
    print(classification_report(best_checkpoint["y_val_true"], best_checkpoint["y_val_pred"], digits=4, zero_division=0))

    print("\nAROUSAL classification report:")
    print(classification_report(best_checkpoint["y_aro_true"], best_checkpoint["y_aro_pred"], digits=4, zero_division=0))

    return {
        "held_out_subject": held_out_subject,
        "best_epoch": best_checkpoint["epoch"],
        "weighted_score": best_checkpoint["weighted_score"],
        "mean_f1": best_checkpoint["mean_f1"],

        "valence_accuracy": best_checkpoint["valence_accuracy"],
        "valence_f1": best_checkpoint["valence_f1"],

        "arousal_accuracy": best_checkpoint["arousal_accuracy"],
        "arousal_f1": best_checkpoint["arousal_f1"],

        "valence_thresh": best_checkpoint["valence_thresh"],
        "arousal_thresh": best_checkpoint["arousal_thresh"],
    }

# =========================================================
# RUN LOSO MULTITASK
# =========================================================
def run_loso_multitask_bottleneck(seq_x_list, y_list, subject_list):
    unique_subjects = sorted(list(set(subject_list)))
    print(f"\nRunning LOSO multitask bottleneck classification across {len(unique_subjects)} subjects:")
    print(unique_subjects)

    all_fold_results = []

    for held_out_subject in unique_subjects:
        fold_result = train_one_loso_fold_multitask_bottleneck(
            held_out_subject,
            seq_x_list,
            y_list,
            subject_list
        )
        all_fold_results.append(fold_result)

    val_accs = np.array([r["valence_accuracy"] for r in all_fold_results], dtype=np.float32)
    val_f1s = np.array([r["valence_f1"] for r in all_fold_results], dtype=np.float32)

    aro_accs = np.array([r["arousal_accuracy"] for r in all_fold_results], dtype=np.float32)
    aro_f1s = np.array([r["arousal_f1"] for r in all_fold_results], dtype=np.float32)

    mean_f1s = np.array([r["mean_f1"] for r in all_fold_results], dtype=np.float32)
    weighted_scores = np.array([r["weighted_score"] for r in all_fold_results], dtype=np.float32)

    summary = {
        "valence_accuracy_mean": float(val_accs.mean()),
        "valence_accuracy_std": float(val_accs.std()),
        "valence_f1_mean": float(val_f1s.mean()),
        "valence_f1_std": float(val_f1s.std()),

        "arousal_accuracy_mean": float(aro_accs.mean()),
        "arousal_accuracy_std": float(aro_accs.std()),
        "arousal_f1_mean": float(aro_f1s.mean()),
        "arousal_f1_std": float(aro_f1s.std()),

        "mean_f1_mean": float(mean_f1s.mean()),
        "mean_f1_std": float(mean_f1s.std()),

        "weighted_score_mean": float(weighted_scores.mean()),
        "weighted_score_std": float(weighted_scores.std()),

        "per_subject": all_fold_results,
    }

    with open("loso_multitask_bottleneck_summary.json", "w") as f:
        json.dump(to_python(summary), f, indent=4)

    print("\n" + "=" * 80)
    print("LOSO MULTITASK BOTTLENECK FINAL SUMMARY")
    print("=" * 80)

    print("\nVALENCE")
    print(f"Accuracy: {summary['valence_accuracy_mean']:.4f} ± {summary['valence_accuracy_std']:.4f}")
    print(f"F1:       {summary['valence_f1_mean']:.4f} ± {summary['valence_f1_std']:.4f}")

    print("\nAROUSAL")
    print(f"Accuracy: {summary['arousal_accuracy_mean']:.4f} ± {summary['arousal_accuracy_std']:.4f}")
    print(f"F1:       {summary['arousal_f1_mean']:.4f} ± {summary['arousal_f1_std']:.4f}")

    print("\nMEAN F1")
    print(f"Mean F1:        {summary['mean_f1_mean']:.4f} ± {summary['mean_f1_std']:.4f}")

    print("\nWEIGHTED SCORE")
    print(f"Weighted Score: {summary['weighted_score_mean']:.4f} ± {summary['weighted_score_std']:.4f}")

    return summary

# =========================================================
# MAIN
# =========================================================
def main():
    print("Loading processed trials...")
    seq_x_list, y_list, subject_list = load_all_trials_with_subjects(DATA_DIR)

    if len(seq_x_list) == 0:
        raise FileNotFoundError(f"No processed trials found in {DATA_DIR}.")

    print("Example x_seq shape:", seq_x_list[0].shape)
    print("Example y shape:", y_list[0].shape)

    config = {
        "model": "MultiTaskBottleneckCNNBiLSTM",
        "cnn_feature_dim": 256,
        "lstm_hidden": 256,
        "dropout": 0.2,
        "lr": LR,
        "epochs": EPOCHS,
        "evaluation": "LOSO multitask binary classification with task-specific bottlenecks",
        "label_strategy": "global train-only threshold per fold",
        "threshold_type": "median",
        "valence_loss_weight": VALENCE_LOSS_WEIGHT,
        "arousal_loss_weight": AROUSAL_LOSS_WEIGHT,
        "selection_score": "0.7 * valence_f1 + 0.3 * arousal_f1",
    }

    with open("loso_multitask_bottleneck_config.json", "w") as f:
        json.dump(to_python(config), f, indent=4)

    summary = run_loso_multitask_bottleneck(seq_x_list, y_list, subject_list)

    with open("loso_multitask_bottleneck_final_summary.json", "w") as f:
        json.dump(to_python(summary), f, indent=4)

    print("\nSaved:")
    print("  loso_multitask_bottleneck_config.json")
    print("  loso_multitask_bottleneck_summary.json")
    print("  loso_multitask_bottleneck_final_summary.json")
    print("  best_model_loso_multitask_bottleneck_<subject>.pt")
    print("  best_y_true_loso_multitask_bottleneck_valence_<subject>.npy")
    print("  best_y_pred_loso_multitask_bottleneck_valence_<subject>.npy")
    print("  best_y_true_loso_multitask_bottleneck_arousal_<subject>.npy")
    print("  best_y_pred_loso_multitask_bottleneck_arousal_<subject>.npy")
    print("  history_loso_multitask_bottleneck_<subject>.json")
    print("  curve_loss_loso_multitask_bottleneck_<subject>.png")
    print("  curve_metrics_loso_multitask_bottleneck_<subject>.png")

if __name__ == "__main__":
    main()

Loading processed trials...
Searching in: /content/drive/MyDrive/processed_meg_regression
Found 82 X files
Total usable trial sequences: 814
Total unique subjects: 21
Example x_seq shape: (4, 143, 2048)
Example y shape: (2,)

Running LOSO multitask bottleneck classification across 21 subjects:
['sub-01', 'sub-02', 'sub-03', 'sub-04', 'sub-05', 'sub-06', 'sub-07', 'sub-08', 'sub-09', 'sub-10', 'sub-11', 'sub-14', 'sub-15', 'sub-16', 'sub-17', 'sub-18', 'sub-19', 'sub-20', 'sub-21', 'sub-22', 'sub-23']

LOSO FOLD | Held-out subject: sub-01 | MULTITASK BOTTLENECK
Train trials: 774
Val trials:   40
Train threshold valence: 0.5000
Train threshold arousal: 0.5000
Train valence counts: [365 409]
Val   valence counts: [20 20]
Train arousal counts: [313 461]
Val   arousal counts: [10 30]
Held-out sub-01 | Epoch 001 | LR: 0.000100 | Train Loss: 1.7190 | Val Loss: 1.6339 | Valence Acc: 0.5000 | Valence F1: 0.6667 | Arousal Acc: 0.7250 | Arousal F1: 0.8406 | Mean F1: 0.7536 | Weighted Score: 0.718

In [ ]:
import zipfile
from google.colab import files

# Define the patterns for the files the user wants to download
file_patterns = [
    'history_loso_multitask_bottleneck_sub-??.json',
    'curve_loss_loso_multitask_bottleneck_sub-??.png',
    'curve_metrics_loso_multitask_bottleneck_sub-??.png'
]

# Collect all files matching the patterns in the current directory
files_to_zip = []
for pattern in file_patterns:
    files_to_zip.extend(glob.glob(pattern))

# Define the name of the zip file
zip_filename = 'loso_multitask_results.zip'

# Create the zip archive
with zipfile.ZipFile(zip_filename, 'w') as zipf:
    for file in files_to_zip:
        if os.path.exists(file):
            zipf.write(file, os.path.basename(file))
        else:
            print(f"[WARN] File not found when zipping: {file}")

print(f"Successfully created {zip_filename} containing {len(files_to_zip)} files.")

# Provide a download link for the zip file
files.download(zip_filename)

Successfully created loso_multitask_results.zip containing 63 files.


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>